# Modelo Estrella — Sala de Detenidos
## Notebook 2 de 2: Construcción del Data Warehouse

### Arquitectura del modelo

```
                    ┌─────────────────┐
                    │   dim_tiempo    │
                    │  id_tiempo (PK) │
                    └────────┬────────┘
                             │
┌──────────────┐    ┌────────┴────────┐    ┌──────────────────┐
│ dim_ubicacion│    │  fact_detenidos │    │  dim_instalacion │
│ id_ubic (PK) ├────┤  id_hecho (PK)  ├────┤  id_inst (PK)    │
└──────────────┘    │  id_tiempo (FK) │    └──────────────────┘
                    │  id_ubic  (FK)  │
┌──────────────┐    │  id_inst  (FK)  │    ┌──────────────────┐
│  dim_reporte │    │  id_rep   (FK)  ├────┤  dim_reporte     │
│  id_rep (PK) ├────┤  [métricas...]  │    │  id_rep (PK)     │
└──────────────┘    └─────────────────┘    └──────────────────┘
```

### Dimensiones construidas
| Dimensión | Descripción | Filas aprox. |
|---|---|---|
| `dim_tiempo` | Calendario completo con atributos temporales | 22 |
| `dim_ubicacion` | Jerarquía RG → Unidad → Sala | ~1.000 |
| `dim_instalacion` | Tipo de instalación (PONAL / URI) | 2 |
| `dim_reporte` | Metadatos del reporte (operador, archivo) | 22 |
| `fact_detenidos` | Tabla de hechos con 30+ métricas | ~46.000 |

---
## ⚙️ CELDA 1 — Configuración

In [ ]:
import os

# ─── AJUSTA ESTAS DOS RUTAS ───────────────────────────────────────────────────
# Ruta al CSV generado por el pipeline (Notebook 1)
RUTA_CSV_STAGING = r'C:\Users\inter\OneDrive - Universidad Externado de Colombia\Tercer Semestre\Big Data\Proyecto_Final\sala_detenidos_consolidado_20260401_1040.csv'

# Carpeta donde se guardarán los CSVs del modelo estrella (para Power BI)
RUTA_DWH = r'C:\Users\inter\OneDrive - Universidad Externado de Colombia\Tercer Semestre\Big Data\Proyecto_Final\DWH'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(RUTA_DWH, exist_ok=True)

if not os.path.isfile(RUTA_CSV_STAGING):
    raise FileNotFoundError(
        f"No se encontró el CSV de staging: {RUTA_CSV_STAGING}\n"
        "Ejecuta primero el Notebook 1 (pipeline) y actualiza la ruta."
    )

print(f"✅ CSV de staging encontrado")
print(f"✅ Carpeta DWH: {RUTA_DWH}")

---
## 📦 CELDA 2 — Importaciones y carga

In [ ]:
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)

# ── Cargar staging ────────────────────────────────────────────────────────────
df = pd.read_csv(RUTA_CSV_STAGING, encoding='utf-8-sig', low_memory=False)

print(f"✅ Staging cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"   Fechas    : {df['FECHA_REPORTE'].min()} → {df['FECHA_REPORTE'].max()}")
print(f"   Operadores: {sorted(df['OPERADOR'].unique())}")
print(f"   RGs       : {sorted(df['RG'].unique())}")
print(f"   Unidades  : {df['UNIDAD'].nunique()} únicas")
print(f"   Ubicaciones: {df['UBICACIÓN SALAS'].nunique()} únicas")

---
## 🗓️ CELDA 3 — `dim_tiempo`

Contiene todos los atributos temporales derivados de la fecha del reporte.
Permite analizar tendencias por día, mes, trimestre y año en Power BI.

In [ ]:
fechas_unicas = pd.to_datetime(
    df['FECHA_REPORTE'].dropna().unique()
).sort_values()

MESES_ES = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
DIAS_ES = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
    4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}

dim_tiempo = pd.DataFrame({
    'id_tiempo'         : range(1, len(fechas_unicas) + 1),
    'fecha'             : fechas_unicas.strftime('%Y-%m-%d'),
    'anio'              : fechas_unicas.year,
    'trimestre'         : fechas_unicas.quarter,
    'trimestre_label'   : ['Q' + str(q) + '-' + str(y)
                           for q, y in zip(fechas_unicas.quarter, fechas_unicas.year)],
    'mes_num'           : fechas_unicas.month,
    'mes_nombre'        : [MESES_ES[m] for m in fechas_unicas.month],
    'mes_anio'          : fechas_unicas.strftime('%Y-%m'),
    'semana_anio'       : fechas_unicas.isocalendar().week.values,
    'dia_mes'           : fechas_unicas.day,
    'dia_semana_num'    : fechas_unicas.dayofweek,
    'dia_semana_nombre' : [DIAS_ES[d] for d in fechas_unicas.dayofweek],
    'es_fin_semana'     : (fechas_unicas.dayofweek >= 5).astype(int),
    'es_fin_mes'        : (fechas_unicas.day >= 28).astype(int),
    'periodo'           : [
        'Enero-Junio 2024' if (y == 2024 and m <= 6) else
        'Julio-Diciembre 2024' if (y == 2024 and m > 6) else
        'Enero-Junio 2025' if (y == 2025 and m <= 6) else
        'Julio-Diciembre 2025'
        for y, m in zip(fechas_unicas.year, fechas_unicas.month)
    ],
})

print(f"✅ dim_tiempo: {dim_tiempo.shape[0]} filas × {dim_tiempo.shape[1]} columnas")
display(dim_tiempo)

---
## 📍 CELDA 4 — `dim_ubicacion`

Captura la jerarquía geográfica completa: **RG → Unidad → Sala**.
Permite drill-down desde región hasta sala individual en Power BI.

In [ ]:
# Descripción de cada RG (Regional de Policía)
DESC_RG = {
    'REMSA' : 'Región Metropolitana de Seguridad y Apoyo',
    'RG 1'  : 'Regional 1 — Noroccidente',
    'RG 2'  : 'Regional 2 — Nororiente',
    'RG 3'  : 'Regional 3 — Centro',
    'RG 4'  : 'Regional 4 — Occidente',
    'RG 5'  : 'Regional 5 — Suroccidente',
    'RG 6'  : 'Regional 6 — Suroriente',
    'RG 7'  : 'Regional 7 — Caribe',
    'RG 8'  : 'Regional 8 — Llanos y Amazonia',
}

# Extraer combinaciones únicas RG + UNIDAD + UBICACIÓN SALAS
dim_ubicacion = (
    df[['RG', 'UNIDAD', 'UBICACIÓN SALAS']]
    .drop_duplicates()
    .sort_values(['RG', 'UNIDAD', 'UBICACIÓN SALAS'])
    .reset_index(drop=True)
)

# Limpiar espacios en UBICACIÓN SALAS
dim_ubicacion['UBICACIÓN SALAS'] = dim_ubicacion['UBICACIÓN SALAS'].str.strip()

# Agregar descripción del RG
dim_ubicacion.insert(0, 'id_ubicacion', range(1, len(dim_ubicacion) + 1))
dim_ubicacion['rg_descripcion'] = dim_ubicacion['RG'].map(DESC_RG).fillna('Desconocido')

# Renombrar para consistencia
dim_ubicacion = dim_ubicacion.rename(columns={
    'RG'              : 'rg_codigo',
    'UNIDAD'          : 'unidad_codigo',
    'UBICACIÓN SALAS' : 'sala_nombre',
})

# Indicador si es metropolitana o departamental
dim_ubicacion['tipo_unidad'] = dim_ubicacion['unidad_codigo'].apply(
    lambda u: 'Metropolitana' if str(u).startswith('ME') else
              'Comando' if str(u).startswith('CO') else
              'Departamental'
)

print(f"✅ dim_ubicacion: {dim_ubicacion.shape[0]:,} filas × {dim_ubicacion.shape[1]} columnas")
print(f"   RGs únicas     : {dim_ubicacion['rg_codigo'].nunique()}")
print(f"   Unidades únicas: {dim_ubicacion['unidad_codigo'].nunique()}")
print(f"   Salas únicas   : {dim_ubicacion['sala_nombre'].nunique()}")
print(f"\n   Distribución por tipo:")
print(dim_ubicacion['tipo_unidad'].value_counts().to_string())
display(dim_ubicacion.head(10))

---
## 🏢 CELDA 5 — `dim_instalacion`

Tipo de instalación donde están los detenidos. Solo dos valores pero
fundamentales para segmentar todos los análisis en Power BI.

In [ ]:
dim_instalacion = pd.DataFrame({
    'id_instalacion'  : [1, 2],
    'codigo'          : ['PONAL', 'URI'],
    'nombre'          : [
        'Instalaciones Policiales',
        'Unidad de Reacción Inmediata (URI)'
    ],
    'descripcion'     : [
        'Salas de detención en estaciones y subestaciones de Policía Nacional',
        'Salas de detención transitoria de la Fiscalía General de la Nación'
    ],
    'entidad_cargo'   : ['Policía Nacional', 'Fiscalía General de la Nación'],
})

print(f"✅ dim_instalacion: {dim_instalacion.shape[0]} filas × {dim_instalacion.shape[1]} columnas")
display(dim_instalacion)

---
## 📋 CELDA 6 — `dim_reporte`

Metadatos de cada reporte diario. Permite filtrar y auditar por
operador responsable del diligenciamiento.

In [ ]:
dim_reporte = (
    df[['ARCHIVO_ORIGEN', 'FECHA_REPORTE', 'OPERADOR']]
    .drop_duplicates()
    .sort_values('FECHA_REPORTE')
    .reset_index(drop=True)
)

dim_reporte.insert(0, 'id_reporte', range(1, len(dim_reporte) + 1))

# Enriquecer con atributos adicionales del operador
RANGO_OPERADOR = {
    'AYALA'  : 'IT. — Intendente',
    'DÁVILA' : 'PT. — Patrullero',
    'ACOSTA' : 'IT. — Intendente',
    'GARCIA' : 'SI. — Subintendente',
    'TOBÓN'  : 'IJ. — Intendente Jefe',
}
dim_reporte['operador_rango'] = dim_reporte['OPERADOR'].map(RANGO_OPERADOR).fillna('Desconocido')

# Renombrar
dim_reporte = dim_reporte.rename(columns={
    'ARCHIVO_ORIGEN' : 'archivo_origen',
    'FECHA_REPORTE'  : 'fecha_reporte',
    'OPERADOR'       : 'operador',
})

print(f"✅ dim_reporte: {dim_reporte.shape[0]} filas × {dim_reporte.shape[1]} columnas")
display(dim_reporte)

---
## ⭐ CELDA 7 — `fact_detenidos`

Tabla de hechos central. Cada fila representa una **sala en una fecha específica**
con todas sus métricas para PONAL y URI.

El modelo convierte el CSV ancho (63 columnas) en una tabla de hechos
normalizada con claves foráneas a todas las dimensiones y métricas limpias.

In [ ]:
# ── PASO 1: Crear copia de trabajo y limpiar espacios ────────────────────────
fact = df.copy()
fact['UBICACIÓN SALAS'] = fact['UBICACIÓN SALAS'].str.strip()
fact['UNIDAD']          = fact['UNIDAD'].str.strip()

# ── PASO 2: Mapear claves foráneas ───────────────────────────────────────────

# FK → dim_tiempo
fecha_a_id = dict(zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo']))
fact['id_tiempo'] = fact['FECHA_REPORTE'].map(fecha_a_id)

# FK → dim_ubicacion
ubic_key = dim_ubicacion.set_index(
    ['rg_codigo', 'unidad_codigo', 'sala_nombre']
)['id_ubicacion']
fact['id_ubicacion'] = (
    pd.MultiIndex.from_arrays([fact['RG'], fact['UNIDAD'], fact['UBICACIÓN SALAS']])
    .map(ubic_key)
)

# FK → dim_reporte
rep_key = dict(zip(dim_reporte['fecha_reporte'], dim_reporte['id_reporte']))
fact['id_reporte'] = fact['FECHA_REPORTE'].map(rep_key)

# ── PASO 3: Definir mapeo exacto de columnas → métricas ─────────────────────
# Cada tupla: (nombre_columna_csv, nombre_metrica_fact)
METRICAS_PONAL = [
    ('TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI',
     'ponal_total_detenidos'),
    ('TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI',
     'ponal_total_personal_custodia'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO_CANTIDAD DE SALAS DEPTO O METRO',
     'ponal_cantidad_salas'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO_CAPACIDAD DE LAS SALAS DEPTO O METRO',
     'ponal_capacidad_salas'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES_IMPUTADOS CON MAS DE 12 MESES',
     'ponal_imputados_mas_12_meses'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS_TOTAL DE PERSONAS EN LAS SALAS',
     'ponal_total_personas_salas'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS',
     'ponal_condenados'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° IMPUTADOS',
     'ponal_imputados'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_CUANTOS NO TIENEN LA DOCUMENTACIÓN COMPLETA PARA TRASLADO A CENTRO CARCELARIO',
     'ponal_sin_documentacion'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_CUANTOS LLEVAN MAS DE 36 HORAS',
     'ponal_mas_36_horas'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_# DE PERSONAS CON MEDIDA NO INTRA MURAL (CASA POR CÁRCEL), QUE SE ENCUENTRAN AÚN EN LAS INSTALACIONES',
     'ponal_casa_por_carcel'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_GENERO_M',
     'ponal_genero_m'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_GENERO_F',
     'ponal_genero_f'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_GENERO_LGBTI',
     'ponal_genero_lgbti'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_OF',
     'ponal_personal_of'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_M/E',
     'ponal_personal_me'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_PT/AG',
     'ponal_personal_ptag'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_AUX',
     'ponal_personal_aux'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL',
     'ponal_personal_total'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)',
     'ponal_condiciones_medicas'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_ALIMENTACIÓN A CARGO DE_ALCALDÍA',
     'ponal_alim_alcaldia'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_ALIMENTACIÓN A CARGO DE_POLICÍA',
     'ponal_alim_policia'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_ALIMENTACIÓN A CARGO DE_INPEC',
     'ponal_alim_inpec'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_ALIMENTACIÓN A CARGO DE_FAMILIARES',
     'ponal_alim_familiares'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_# DE EXTRANJEROS EN INSTALACIONES POLICIALES_# DE EXTRANJEROS EN INSTALACIONES POLICIALES',
     'ponal_extranjeros'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_# DE EXTRANJEROS ((VENEZOLANOS))_# DE EXTRANJEROS ((VENEZOLANOS))',
     'ponal_venezolanos'),
    ('DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS (VENEZOLANOS))_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS (VENEZOLANOS))',
     'ponal_condiciones_medicas_venezolanos'),
]

METRICAS_URI = [
    ('DETENIDOS EN INSTALACIONES "URI"_CANTIDAD DE SALAS DEPTO O METRO_CANTIDAD DE SALAS DEPTO O METRO',
     'uri_cantidad_salas'),
    ('DETENIDOS EN INSTALACIONES "URI"_CAPACIDAD DE LAS SALAS DEPTO O METRO_CAPACIDAD DE LAS SALAS DEPTO O METRO',
     'uri_capacidad_salas'),
    ('DETENIDOS EN INSTALACIONES "URI"_IMPUTADOS CON MAS DE 12 MESES_IMPUTADOS CON MAS DE 12 MESES',
     'uri_imputados_mas_12_meses'),
    ('DETENIDOS EN INSTALACIONES "URI"_TOTAL DE PERSONAS EN LAS SALAS_TOTAL DE PERSONAS EN LAS SALAS',
     'uri_total_personas_salas'),
    ('DETENIDOS EN INSTALACIONES "URI"_CONDICIÓN_N° CONDENADOS',
     'uri_condenados'),
    ('DETENIDOS EN INSTALACIONES "URI"_CONDICIÓN_N° IMPUTADOS',
     'uri_imputados'),
    ('DETENIDOS EN INSTALACIONES "URI"_CONDICIÓN_CUANTOS LLEVAN MAS DE 36 HORAS',
     'uri_mas_36_horas'),
    ('DETENIDOS EN INSTALACIONES "URI"_CONDICIÓN_# DE PERSONAS CON MEDIDA NO INTRA MURAL (CASA POR CÁRCEL), QUE SE ENCUENTRAN AÚN EN LAS INSTALACIONES',
     'uri_casa_por_carcel'),
    ('DETENIDOS EN INSTALACIONES "URI"_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)',
     'uri_condiciones_medicas'),
    ('DETENIDOS EN INSTALACIONES "URI"_GENERO_M',
     'uri_genero_m'),
    ('DETENIDOS EN INSTALACIONES "URI"_GENERO_F',
     'uri_genero_f'),
    ('DETENIDOS EN INSTALACIONES "URI"_GENERO_LGBTI',
     'uri_genero_lgbti'),
    ('DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_OF',
     'uri_personal_of'),
    ('DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_M/E',
     'uri_personal_me'),
    ('DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_PT/AG',
     'uri_personal_ptag'),
    ('DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_AUX',
     'uri_personal_aux'),
    ('DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL',
     'uri_personal_total'),
    ('DETENIDOS EN INSTALACIONES "URI"_# DE EXTRANJEROS EN URI_# DE EXTRANJEROS EN URI',
     'uri_extranjeros'),
    ('DETENIDOS EN INSTALACIONES "URI"_# DE EXTRANJEROS ((VENEZOLANOS))_# DE EXTRANJEROS ((VENEZOLANOS))',
     'uri_venezolanos'),
    ('DETENIDOS EN INSTALACIONES "URI"_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS (VENEZOLANOS))_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS (VENEZOLANOS))',
     'uri_condiciones_medicas_venezolanos'),
]

# ── PASO 4: Construir fact_detenidos ─────────────────────────────────────────
cols_fact = ['id_tiempo', 'id_ubicacion', 'id_reporte']
fact_detenidos = fact[cols_fact].copy()
fact_detenidos.insert(0, 'id_hecho', range(1, len(fact_detenidos) + 1))

# Agregar métricas PONAL
for col_csv, col_fact in METRICAS_PONAL:
    if col_csv in fact.columns:
        fact_detenidos[col_fact] = pd.to_numeric(fact[col_csv], errors='coerce').fillna(0).astype(int)
    else:
        fact_detenidos[col_fact] = 0
        print(f"  ⚠️  No encontrada: {col_csv[:60]}")

# Agregar métricas URI
for col_csv, col_fact in METRICAS_URI:
    if col_csv in fact.columns:
        fact_detenidos[col_fact] = pd.to_numeric(fact[col_csv], errors='coerce').fillna(0).astype(int)
    else:
        fact_detenidos[col_fact] = 0
        print(f"  ⚠️  No encontrada: {col_csv[:60]}")

# ── PASO 5: Métricas calculadas (KPIs derivados) ─────────────────────────────
# Tasa de hacinamiento: personas / capacidad (>1 = hacinado)
fact_detenidos['ponal_tasa_hacinamiento'] = (
    fact_detenidos['ponal_total_personas_salas'] /
    fact_detenidos['ponal_capacidad_salas'].replace(0, np.nan)
).round(3).fillna(0)

fact_detenidos['uri_tasa_hacinamiento'] = (
    fact_detenidos['uri_total_personas_salas'] /
    fact_detenidos['uri_capacidad_salas'].replace(0, np.nan)
).round(3).fillna(0)

# Razón guardián/detenido (cuántos detenidos por cada custodio)
fact_detenidos['ponal_ratio_guardian_detenido'] = (
    fact_detenidos['ponal_total_personas_salas'] /
    fact_detenidos['ponal_personal_total'].replace(0, np.nan)
).round(2).fillna(0)

# Total detenidos consolidado (PONAL + URI)
fact_detenidos['total_detenidos_consolidado'] = (
    fact_detenidos['ponal_total_personas_salas'] +
    fact_detenidos['uri_total_personas_salas']
)

# Porcentaje imputados vs condenados PONAL
total_cond = fact_detenidos['ponal_condenados'] + fact_detenidos['ponal_imputados']
fact_detenidos['ponal_pct_imputados'] = (
    fact_detenidos['ponal_imputados'] / total_cond.replace(0, np.nan) * 100
).round(1).fillna(0)

# Flag de alerta: salas con detenidos mas de 36 horas > 0
fact_detenidos['alerta_mas_36h'] = (
    (fact_detenidos['ponal_mas_36_horas'] > 0) |
    (fact_detenidos['uri_mas_36_horas'] > 0)
).astype(int)

# Flag de alerta: hacinamiento
fact_detenidos['alerta_hacinamiento'] = (
    fact_detenidos['ponal_tasa_hacinamiento'] > 1.0
).astype(int)

print(f"✅ fact_detenidos: {fact_detenidos.shape[0]:,} filas × {fact_detenidos.shape[1]} columnas")
print(f"\nColumnas generadas:")
for i, col in enumerate(fact_detenidos.columns):
    print(f"  [{i:2d}] {col}")

# Verificar integridad referencial
nan_fks = fact_detenidos[['id_tiempo','id_ubicacion','id_reporte']].isnull().sum()
print(f"\n🔍 Integridad referencial (NaN en FKs):")
print(nan_fks.to_string())

---
## 🔍 CELDA 8 — Validación del modelo

In [ ]:
print("═" * 65)
print("VALIDACIÓN DEL MODELO ESTRELLA")
print("═" * 65)

print(f"\n📊 Resumen de tablas:")
tablas = {
    'dim_tiempo'      : dim_tiempo,
    'dim_ubicacion'   : dim_ubicacion,
    'dim_instalacion' : dim_instalacion,
    'dim_reporte'     : dim_reporte,
    'fact_detenidos'  : fact_detenidos,
}
for nombre, tabla in tablas.items():
    print(f"  {nombre:<22} {tabla.shape[0]:>7,} filas × {tabla.shape[1]:>3} columnas")

print(f"\n📊 KPIs globales del dataset:")
total_det  = fact_detenidos['ponal_total_personas_salas'].sum()
total_uri  = fact_detenidos['uri_total_personas_salas'].sum()
hac_count  = fact_detenidos['alerta_hacinamiento'].sum()
h36_count  = fact_detenidos['alerta_mas_36h'].sum()
venz_total = fact_detenidos['ponal_venezolanos'].sum() + fact_detenidos['uri_venezolanos'].sum()

print(f"  Total detenidos PONAL (suma histórica) : {total_det:>10,}")
print(f"  Total detenidos URI   (suma histórica) : {total_uri:>10,}")
print(f"  Registros con hacinamiento (PONAL>1)   : {hac_count:>10,}")
print(f"  Registros con +36h detenidos           : {h36_count:>10,}")
print(f"  Total venezolanos (suma histórica)     : {venz_total:>10,}")

print(f"\n📊 Top 5 unidades por promedio de detenidos:")
top_unidades = (
    fact_detenidos
    .merge(dim_ubicacion[['id_ubicacion','unidad_codigo','sala_nombre']], on='id_ubicacion')
    .groupby('unidad_codigo')['ponal_total_personas_salas']
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .round(1)
)
print(top_unidades.to_string())

print(f"\n✅ Modelo estrella validado correctamente.")

---
## 💾 CELDA 9 — Exportar tablas del DWH

Exporta cada tabla como CSV independiente en la carpeta `DWH/`.
Power BI se conecta a esta carpeta y carga las 5 tablas directamente.

In [ ]:
from datetime import datetime

tablas_exportar = {
    'dim_tiempo'      : dim_tiempo,
    'dim_ubicacion'   : dim_ubicacion,
    'dim_instalacion' : dim_instalacion,
    'dim_reporte'     : dim_reporte,
    'fact_detenidos'  : fact_detenidos,
}

print("Exportando tablas del modelo estrella...\n")

for nombre, tabla in tablas_exportar.items():
    ruta_tabla = os.path.join(RUTA_DWH, f'{nombre}.csv')
    tabla.to_csv(ruta_tabla, index=False, encoding='utf-8-sig')
    tam_kb = os.path.getsize(ruta_tabla) / 1024
    print(f"  ✅ {nombre:<22} → {nombre}.csv  "
          f"({tabla.shape[0]:>7,} filas, {tam_kb:>7.1f} KB)")

print(f"\n📁 Todos los archivos guardados en:")
print(f"   {RUTA_DWH}")
print(f"\n💡 Instrucciones para Power BI:")
print(f"   1. Abrir Power BI Desktop")
print(f"   2. Inicio → Obtener datos → Texto/CSV")
print(f"   3. Cargar los 5 archivos CSV de la carpeta DWH/")
print(f"   4. En Modelo → crear relaciones:")
print(f"      fact_detenidos[id_tiempo]    → dim_tiempo[id_tiempo]")
print(f"      fact_detenidos[id_ubicacion] → dim_ubicacion[id_ubicacion]")
print(f"      fact_detenidos[id_reporte]   → dim_reporte[id_reporte]")

---
## 🔄 CELDA 10 — Actualización incremental (uso futuro)

Cuando el pipeline genere un nuevo CSV con más fechas, ejecuta solo
esta celda para agregar los nuevos hechos sin reconstruir todo el modelo.

In [ ]:
# ─── AJUSTAR CUANDO HAYA NUEVOS DATOS ────────────────────────────────────────
RUTA_CSV_NUEVO = ''  # ← poner aquí la ruta del nuevo CSV del pipeline
# ─────────────────────────────────────────────────────────────────────────────

if not RUTA_CSV_NUEVO:
    print("ℹ️  No hay CSV nuevo definido. Ajusta RUTA_CSV_NUEVO para usar esta celda.")
else:
    df_nuevo = pd.read_csv(RUTA_CSV_NUEVO, encoding='utf-8-sig', low_memory=False)

    # Filtrar solo fechas que no existen aún en dim_tiempo
    fechas_existentes = set(dim_tiempo['fecha'])
    fechas_nuevas     = set(df_nuevo['FECHA_REPORTE'].unique()) - fechas_existentes

    if not fechas_nuevas:
        print("ℹ️  No hay fechas nuevas en el CSV. El modelo ya está actualizado.")
    else:
        print(f"✅ {len(fechas_nuevas)} fechas nuevas detectadas: {sorted(fechas_nuevas)}")
        print("   Ejecuta las celdas 3-9 con df = pd.concat([df, df_nuevo]) para actualizar.")